# 02 Evaluation

Purpose: Evaluate all scoring variants (including KDE-DNDS) on the test set, run alpha and KDE-bandwidth sensitivity analyses, and save summary artifacts.

Inputs: `dataset/CVPR_2024_dataset_Test/`, `dataset_text/test.csv`, `models/mobilenetv3_best.pth`, `text_models/distilbert_best.pth`, populated `chroma_db/`.

Outputs: `results/phase2/evaluation_results.json`, `figures/phase2/scoring_comparison.png`, `figures/phase2/alpha_sensitivity.png`, `figures/phase2/kde_bandwidth_ablation.png`, `figures/phase2/confusion_matrices_phase2.png`, `figures/phase2/phase2_vs_phase1.png`.

In [1]:
import importlib
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

from src.phase2.config import get_phase2_config
from src.phase2.data_utils import build_records_from_csv
from src.phase2.db_client import (
    get_class_counts,
    get_image_collection,
    get_persistent_client,
    get_text_collection,
)
import src.phase2.evaluation as phase2_evaluation

importlib.reload(phase2_evaluation)
from src.phase2.evaluation import (
    evaluate_variant,
    save_alpha_sweep_csv,
    save_metrics_summary_csv,
    save_predictions_csv,
    save_results,
)
from src.phase2.gpu_utils import (
    clear_gpu_memory,
    get_device,
    print_device_info,
    print_gpu_memory,
)
from src.phase2.imbalance import infer_class_groups
from src.phase2.scoring import (
    global_dnds,
    idw,
    kde_dnds,
    local_dnds,
    majority_vote,
    traditional,
)
from src.phase2.traditional import load_phase1_traditional_components
from src.phase2.visualization import (
    plot_alpha_sensitivity,
    plot_confusion_matrices,
    plot_kde_bandwidth_ablation,
    plot_phase2_vs_phase1,
    plot_scoring_comparison,
)

CONFIG = get_phase2_config()

# Per-notebook GPU controls
PREFER_GPU = True
USE_HALF_PRECISION = False
CLEANUP_INTERVAL = 0
MEMORY_LOG_INTERVAL = 0

# Hyperparameter sweep controls
PARALLEL_SWEEPS = True
SWEEP_MAX_WORKERS = 20

DEVICE = get_device(prefer_gpu=PREFER_GPU)

REPO_ROOT = Path("../..").resolve()
TEST_DIR = REPO_ROOT / "dataset" / "CVPR_2024_dataset_Test"
TEST_CSV = REPO_ROOT / "dataset_text" / "test.csv"
RESULTS_PATH = REPO_ROOT / "results" / "phase2" / "evaluation_results.json"
METRICS_CSV_PATH = REPO_ROOT / "results" / "phase2" / "metrics_summary.csv"
ALPHA_CSV_PATH = REPO_ROOT / "results" / "phase2" / "alpha_sweep.csv"
PREDICTIONS_CSV_PATH = (
    REPO_ROOT / "results" / "phase2" / "scoring_variants_predictions.csv"
)
FIGURES_DIR = REPO_ROOT / "figures" / "phase2"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not TEST_DIR.exists() or not TEST_CSV.exists():
    raise FileNotFoundError(
        "Test dataset missing. Expected dataset and dataset_text paths in repo root."
    )

print_device_info(DEVICE)
if MEMORY_LOG_INTERVAL > 0:
    print_gpu_memory(prefix="Startup GPU memory", device=DEVICE)

d:\University of Calgary\Winter 2026\ENSF617-Introduction-To-Machine-Learning\trash-classification-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU (4.00 GB VRAM, allocated 0.00 GB, reserved 0.00 GB)


In [2]:
test_samples, missing_examples, total_rows = build_records_from_csv(
    csv_path=TEST_CSV,
    split_dir=TEST_DIR,
    text_column="text",
    label_column="label",
    text_key="text",
)

if missing_examples:
    print("Skipped test rows with missing image files (up to 10 shown):")
    for item in missing_examples:
        print(f"  - {item}")

if not test_samples:
    raise RuntimeError("No test samples found after image path resolution.")

print(f"Test samples available for evaluation: {len(test_samples)} / {total_rows}")

client = get_persistent_client(str(REPO_ROOT / "chroma_db"))
image_collection = get_image_collection(client)
text_collection = get_text_collection(client)

image_class_counts = get_class_counts(image_collection)
majority_classes, minority_classes = infer_class_groups(
    image_class_counts, threshold=float(CONFIG["majority_threshold"])
)
CONFIG["majority_classes"] = majority_classes
CONFIG["minority_classes"] = minority_classes
print(
    f"Dynamic class grouping -> majority: {majority_classes}, minority: {minority_classes}"
)

variant_fns = {
    "majority_vote": majority_vote,
    "idw": idw,
    "global_dnds": global_dnds,
    "local_dnds": local_dnds,
    "kde_dnds": kde_dnds,
}

Test samples available for evaluation: 3452 / 3452


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 808.91it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dynamic class grouping -> majority: ['Blue'], minority: ['Black', 'Green', 'TTR']


In [ ]:
results = {
    "variants": {},
    "alpha_sweep": {
        "local_dnds": {"alphas": [], "macro_f1": []},
        "kde_dnds": {"alphas": [], "macro_f1": []},
    },
    "kde_bandwidth": {"bandwidths": [], "macro_f1": []},
    "k_density_sweep": {"values": [], "macro_f1": []},
}


def _evaluate_for_sweep(
    variant_fn,
    *,
    alpha=None,
    config_overrides=None,
    score_kwargs=None,
):
    eval_config = dict(CONFIG)
    if config_overrides:
        eval_config.update(config_overrides)
    return evaluate_variant(
        variant_fn,
        test_samples,
        image_collection,
        text_collection,
        eval_config,
        alpha=alpha,
        score_kwargs=score_kwargs,
        cleanup_interval=CLEANUP_INTERVAL,
        memory_log_interval=MEMORY_LOG_INTERVAL,
    )


def _execute_jobs(jobs):
    if PARALLEL_SWEEPS and len(jobs) > 1:
        workers = max(1, min(SWEEP_MAX_WORKERS, len(jobs)))
        print(f"Running sweep in parallel with {workers} worker threads...")
        with ThreadPoolExecutor(max_workers=workers) as pool:
            futures = [pool.submit(job) for job in jobs]
            return [future.result() for future in futures]
    return [job() for job in jobs]


for name, fn in variant_fns.items():
    metrics = _evaluate_for_sweep(fn)
    results["variants"][name] = metrics

# Alpha sweeps for local_dnds and kde_dnds
alpha_jobs = []
for alpha in CONFIG["alpha_sweep"]:
    alpha_value = float(alpha)

    def _local_alpha_job(a=alpha_value):
        metrics = _evaluate_for_sweep(local_dnds, alpha=a)
        return "local_dnds", a, metrics["macro_f1"]

    def _kde_alpha_job(a=alpha_value):
        metrics = _evaluate_for_sweep(kde_dnds, alpha=a)
        return "kde_dnds", a, metrics["macro_f1"]

    alpha_jobs.extend([_local_alpha_job, _kde_alpha_job])

alpha_points = _execute_jobs(alpha_jobs)
local_points = sorted(
    [(a, score) for name, a, score in alpha_points if name == "local_dnds"],
    key=lambda item: item[0],
)
kde_points = sorted(
    [(a, score) for name, a, score in alpha_points if name == "kde_dnds"],
    key=lambda item: item[0],
)
results["alpha_sweep"]["local_dnds"]["alphas"] = [a for a, _ in local_points]
results["alpha_sweep"]["local_dnds"]["macro_f1"] = [score for _, score in local_points]
results["alpha_sweep"]["kde_dnds"]["alphas"] = [a for a, _ in kde_points]
results["alpha_sweep"]["kde_dnds"]["macro_f1"] = [score for _, score in kde_points]

best_alpha_local = max(local_points, key=lambda item: item[1])[0]
best_alpha_kde = max(kde_points, key=lambda item: item[1])[0]
results["best_alpha_local"] = best_alpha_local
results["best_alpha_kde"] = best_alpha_kde

# Local DNDS K_density sweep (using best local alpha from alpha sweep)
k_density_jobs = []
for k_value in CONFIG.get("K_density_sweep", [CONFIG["K_density"]]):
    k_density = int(k_value)

    def _k_density_job(k=k_density):
        metrics = _evaluate_for_sweep(
            local_dnds,
            alpha=best_alpha_local,
            config_overrides={"K_density": k},
        )
        return k, metrics["macro_f1"]

    k_density_jobs.append(_k_density_job)

k_density_points = sorted(_execute_jobs(k_density_jobs), key=lambda item: item[0])
results["k_density_sweep"]["values"] = [k for k, _ in k_density_points]
results["k_density_sweep"]["macro_f1"] = [score for _, score in k_density_points]
best_k_density = max(k_density_points, key=lambda item: item[1])[0]
results["best_k_density_local"] = int(best_k_density)
CONFIG["K_density"] = int(best_k_density)

# Overwrite local_dnds result with tuned hyperparameters.
results["variants"]["local_dnds"] = _evaluate_for_sweep(
    local_dnds,
    alpha=best_alpha_local,
    config_overrides={"K_density": int(best_k_density)},
)

# KDE bandwidth ablation
kde_bandwidth_jobs = []
for bandwidth in CONFIG["kde_bandwidth_sweep"]:
    bw = float(bandwidth)

    def _kde_bandwidth_job(b=bw):
        metrics = _evaluate_for_sweep(
            kde_dnds,
            alpha=best_alpha_kde,
            score_kwargs={"bandwidth": b},
        )
        return b, metrics["macro_f1"]

    kde_bandwidth_jobs.append(_kde_bandwidth_job)

kde_bandwidth_points = sorted(
    _execute_jobs(kde_bandwidth_jobs), key=lambda item: item[0]
)
results["kde_bandwidth"]["bandwidths"] = [b for b, _ in kde_bandwidth_points]
results["kde_bandwidth"]["macro_f1"] = [score for _, score in kde_bandwidth_points]
best_bw_idx = max(
    range(len(results["kde_bandwidth"]["macro_f1"])),
    key=lambda i: results["kde_bandwidth"]["macro_f1"][i],
)
results["best_kde_bandwidth"] = results["kde_bandwidth"]["bandwidths"][best_bw_idx]
CONFIG["kde_bandwidth"] = float(results["best_kde_bandwidth"])

print(f"Best local DNDS alpha from sweep: {best_alpha_local}")
print(f"Best local DNDS K_density from sweep: {best_k_density}")
print(f"Best KDE-DNDS alpha from sweep: {best_alpha_kde}")
print(f"Best KDE bandwidth from sweep: {results['best_kde_bandwidth']}")

Evaluating kde_dnds: 100%|██████████| 3452/3452 [38:49<00:00,  1.48it/s]


Running sweep in parallel with 14 worker threads...


Evaluating local_dnds:   0%|          | 0/3452 [00:00<?, ?it/s]













































































Evaluating local_dnds:   0%|          | 1/3452 [00:00<49:37,  1.16it/s]




































Evaluating local_dnds:   0%|          | 2/3452 [00:02<59:30,  1.03s/it]




































Evaluating local_dnds:   0%|          | 3/3452 [00:03<1:08:44,  1.20s/it]






































Evaluating local_dnds:   0%|          | 4/3452 [00:05<1:22:28,  1.44s/it]











































































Evaluating local_dnds:   0%|          | 5/3452 [00:06<1:24:41,  1.47s/it]










































Evaluating local_dnds:   0%|          | 6/3452 [00:07<1:15:46,  1.32s/it]






































Evaluating local_dnds:   0%|          | 7/3452 [00:09<1:14:18,  1.29s/it]











































































Evalu

In [ ]:
# Traditional baseline
image_ckpt = REPO_ROOT / "models" / "mobilenetv3_best.pth"
text_ckpt = REPO_ROOT / "text_models" / "distilbert_best.pth"

if image_ckpt.exists() and text_ckpt.exists():
    clear_gpu_memory()
    components = load_phase1_traditional_components(
        image_checkpoint_path=image_ckpt,
        text_checkpoint_path=text_ckpt,
        num_classes=len(CONFIG["class_names"]),
        device=DEVICE,
        use_half_precision=USE_HALF_PRECISION,
    )

    traditional_metrics = evaluate_variant(
        traditional,
        test_samples,
        image_collection,
        text_collection,
        CONFIG,
        score_kwargs=components,
        cleanup_interval=CLEANUP_INTERVAL,
        memory_log_interval=MEMORY_LOG_INTERVAL,
    )
    results["variants"]["traditional"] = traditional_metrics

    # Release model references and clear CUDA cache after baseline evaluation.
    del components
    clear_gpu_memory()
else:
    print("Traditional checkpoints missing; skipping traditional baseline run.")

In [ ]:
best_alpha_local_idx = max(
    range(len(results["alpha_sweep"]["local_dnds"]["macro_f1"])),
    key=lambda i: results["alpha_sweep"]["local_dnds"]["macro_f1"][i],
)
best_alpha_kde_idx = max(
    range(len(results["alpha_sweep"]["kde_dnds"]["macro_f1"])),
    key=lambda i: results["alpha_sweep"]["kde_dnds"]["macro_f1"][i],
)
results["best_alpha_local"] = results["alpha_sweep"]["local_dnds"]["alphas"][
    best_alpha_local_idx
]
results["best_alpha_kde"] = results["alpha_sweep"]["kde_dnds"]["alphas"][
    best_alpha_kde_idx
]

save_results(results, str(RESULTS_PATH))
save_metrics_summary_csv(results["variants"], str(METRICS_CSV_PATH))
save_alpha_sweep_csv(results["alpha_sweep"]["local_dnds"], str(ALPHA_CSV_PATH))
save_predictions_csv(results["variants"], str(PREDICTIONS_CSV_PATH))

plot_scoring_comparison(results, str(FIGURES_DIR / "scoring_comparison.png"))
plot_alpha_sensitivity(
    results["alpha_sweep"], str(FIGURES_DIR / "alpha_sensitivity.png")
)
plot_kde_bandwidth_ablation(
    results["kde_bandwidth"], str(FIGURES_DIR / "kde_bandwidth_ablation.png")
)
plot_confusion_matrices(
    results,
    CONFIG["class_names"],
    str(FIGURES_DIR / "confusion_matrices_phase2.png"),
    variants=[
        "majority_vote",
        "idw",
        "global_dnds",
        "local_dnds",
        "kde_dnds",
        "traditional",
    ],
)
phase1_macro_f1 = 0.8177
best_phase2 = max(
    v.get("macro_f1", 0.0) for k, v in results["variants"].items() if k != "traditional"
)
plot_phase2_vs_phase1(
    best_phase2, phase1_macro_f1, str(FIGURES_DIR / "phase2_vs_phase1.png")
)

print(f"Saved JSON results: {RESULTS_PATH}")
print(f"Saved CSV metrics: {METRICS_CSV_PATH}")
print(f"Saved CSV alpha sweep (local_dnds): {ALPHA_CSV_PATH}")
print(f"Saved CSV predictions: {PREDICTIONS_CSV_PATH}")

In [ ]:
print("=" * 60)
print("PHASE 2 RAC EXPERIMENT - RESULTS SUMMARY")
print("=" * 60)
print("Variant              | Accuracy | Macro F1 | TTR F1 | Latency")
print("---------------------|----------|----------|--------|--------")
for name, metrics in results["variants"].items():
    ttr_f1 = metrics.get("per_class_f1", {}).get("TTR", 0.0)
    print(
        f"{name:<20} | {metrics['accuracy']*100:6.2f}% | {metrics['macro_f1']:.4f} | {ttr_f1:.4f} | {metrics['latency_ms_per_sample']:.2f} ms"
    )
print("=" * 60)
print(f"Best alpha (local DNDS): {results['best_alpha_local']}")
print(f"Best alpha (KDE-DNDS): {results['best_alpha_kde']}")
print(f"Best KDE bandwidth (h): {results['best_kde_bandwidth']}")